In [6]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Ensure output directories exist so the code doesn't crash when saving
os.makedirs('03_Feature_Selection', exist_ok=True)
os.makedirs('dataset', exist_ok=True)

# 1. Load data
data = pd.read_csv('/content/Telco_Customer_Churn_Dataset.csv')

# 2. Fix 'TotalCharges' which contains hidden blank spaces in the raw CSV
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)

# 3. Drop the non-predictive ID column
data.drop(columns=['customerID'], inplace=True)

# 4. Separate features and target
X = data.drop(columns=['Churn'])
# Convert the target variable to binary (1/0)
y = data['Churn'].map({'Yes': 1, 'No': 0})

# 5. One-Hot Encode categorical variables (Crucial fix for scikit-learn)
# This converts text columns into multiple binary columns (e.g., Contract_One_year)
X_encoded = pd.get_dummies(X, drop_first=True)

# 6. Split the data FIRST to avoid data leakage during feature selection
# Using standard 80/20 split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 7. Train Random Forest model to get feature importances (on Training data ONLY)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 8. Extract feature importances
importances = model.feature_importances_
feature_names = X_train.columns

# 9. Create dataframe of features and their importance scores
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Display top 15 features
print("--- Top 15 Features ---")
print(feature_importance_df.head(15))

# 10. Save complete feature importance list to csv
feature_importance_df.to_csv('03_Feature_Selection/feature_importance.csv', index=False)

# 11. Select and save top 15 features for downstream use
top_15_features = feature_importance_df['Feature'].head(15).tolist()
with open('03_Feature_Selection/selected_features.txt', 'w') as f:
    for feat in top_15_features:
        f.write(f"{feat}\n")

# 12. Subset the train and test datasets to keep ONLY the top 15 features
X_train_selected = X_train[top_15_features]
X_test_selected = X_test[top_15_features]

# 13. Save selected splits to CSV files for reuse in Model Training (Task 5)
X_train_selected.to_csv('dataset/selected_X_train.csv', index=False)
X_test_selected.to_csv('dataset/selected_X_test.csv', index=False)
y_train.to_csv('dataset/y_train.csv', index=False)
y_test.to_csv('dataset/y_test.csv', index=False)

print("\nSuccess: Files successfully saved to '03_Feature_Selection/' and 'dataset/' directories.")

/tmp/ipykernel_9277/3999019847.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)


--- Top 15 Features ---
                           Feature  Importance
3                     TotalCharges    0.189701
1                           tenure    0.175710
2                   MonthlyCharges    0.172418
10     InternetService_Fiber optic    0.036053
28  PaymentMethod_Electronic check    0.035302
25               Contract_Two year    0.030435
13              OnlineSecurity_Yes    0.029238
4                      gender_Male    0.027423
26            PaperlessBilling_Yes    0.025295
5                      Partner_Yes    0.024209
19                 TechSupport_Yes    0.023697
24               Contract_One year    0.021924
15                OnlineBackup_Yes    0.021785
6                   Dependents_Yes    0.020815
0                    SeniorCitizen    0.019905

Success: Files successfully saved to '03_Feature_Selection/' and 'dataset/' directories.


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ==========================================
# 1. Data Loading & Preprocessing
# ==========================================
# Load the dataset
data = pd.read_csv('/content/Telco_Customer_Churn_Dataset.csv')

# Clean 'TotalCharges' and drop 'customerID'
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)
data.drop(columns=['customerID'], inplace=True)

# Encode categorical variables and define features/target
X = pd.get_dummies(data.drop(columns=['Churn']), drop_first=True)
y = data['Churn'].map({'Yes': 1, 'No': 0})

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. Extract Top 4 Features
# ==========================================
print("Extracting feature importances...")
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
rf_selector.fit(X_train, y_train)

# Sort features by importance
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_selector.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Get the top 4
top_4_features = feature_importances['Feature'].head(4).tolist()
print(f"Top 4 Selected Features: {top_4_features}\n")

# Filter training and testing sets to ONLY include the top 4 features
X_train_top4 = X_train[top_4_features]
X_test_top4 = X_test[top_4_features]

# ==========================================
# 3. Data Scaling (Crucial for SVM & LogReg)
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_top4)
X_test_scaled = scaler.transform(X_test_top4)

# ==========================================
# 4. Model Initialization
# ==========================================
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

# ==========================================
# 5. Model Training & Evaluation Loop
# ==========================================
print("Training models and calculating metrics...\n")
report_data = []

for model_name, model in models.items():
    # Train the model
    model.fit(X_train_scaled, y_train)

    # Generate predictions
    y_pred = model.predict(X_test_scaled)
    # Predict probabilities for the positive class (Churn = 1)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Save the results
    report_data.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

# ==========================================
# 6. Final Report Generation
# ==========================================
report_df = pd.DataFrame(report_data)

print("-" * 55)
print("              FINAL MODEL EVALUATION REPORT")
print("-" * 55)
# Formatting the output to 4 decimal places for cleaner viewing
print(report_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("-" * 55)

# Optional: Save report to CSV
report_df.to_csv('model_evaluation_report.csv', index=False)

/tmp/ipykernel_9277/2848093037.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)


Extracting feature importances...
Top 4 Selected Features: ['TotalCharges', 'tenure', 'MonthlyCharges', 'InternetService_Fiber optic']

Training models and calculating metrics...

-------------------------------------------------------
              FINAL MODEL EVALUATION REPORT
-------------------------------------------------------
              Model  Accuracy  F1-Score  ROC-AUC
Logistic Regression    0.8041    0.5741   0.8388
      Random Forest    0.7764    0.5347   0.7970
                SVM    0.8013    0.5364   0.7564
-------------------------------------------------------
